In [2]:
import pandas as pd

df = pd.read_csv("dataset/synthetic_logs.csv")
df.head()

,timestamp,source,log_message,target_label,complexity
0,2025-06-27 07:20:25,ModernCRM,nova.osapi_compute.wsgi.server [req-b9718cd8-f...,HTTP Status,bert
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert
2,1/17/2025 1:29,AnalyticsEngine,Unauthorized access to data was attempted,Security Alert,bert
3,2025-07-12 00:24:16,ModernHR,nova.osapi_compute.wsgi.server [req-4895c258-b...,HTTP Status,bert
4,2025-06-02 18:25:23,BillingSystem,nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...,HTTP Status,bert


In [3]:
print(df.source.unique())

['ModernCRM' 'AnalyticsEngine' 'ModernHR' 'BillingSystem' 'ThirdPartyAPI'
 'LegacyCRM']


In [4]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

c:\Users\KIIT0001\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import numpy as np

In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['log_message'].tolist())

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2882.80it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
embeddings[0]

array([-1.02939621e-01,  3.35459411e-02, -2.20260732e-02,  1.55101740e-03,
       -9.86917876e-03, -1.78956270e-01, -6.34409785e-02, -6.01761639e-02,
        2.81109158e-02,  5.99620491e-02, -1.72618348e-02,  1.43363548e-03,
       -1.49560034e-01,  3.15287686e-03, -5.66030927e-02,  2.71685235e-02,
       -1.49891041e-02, -3.54037657e-02, -3.62936445e-02, -1.45410765e-02,
       -5.61491773e-03,  8.75539035e-02,  4.55120578e-02,  2.50963885e-02,
        1.00187510e-02,  1.24267349e-02, -1.39923573e-01,  7.68696293e-02,
        3.14095505e-02, -4.15247958e-03,  4.36902344e-02,  1.71250012e-02,
       -8.00951198e-02,  5.74006326e-02,  1.89091656e-02,  8.55262503e-02,
        3.96398641e-02, -1.34371817e-01, -1.44360063e-03,  3.06704035e-03,
        1.76854044e-01,  4.44885530e-03, -1.69274509e-02,  2.24266481e-02,
       -4.35049310e-02,  6.09034160e-03, -9.98169929e-03, -6.23972900e-02,
        1.07372422e-02, -6.04895083e-03, -7.14660808e-02, -8.45799781e-03,
       -3.18019874e-02, -

In [8]:
dbscan = DBSCAN(eps=0.2, min_samples=1, metric='cosine')
clusters = dbscan.fit_predict(embeddings)

In [9]:
df['cluster'] = clusters
print(df.head())

             timestamp           source  \
0  2025-06-27 07:20:25        ModernCRM   
1      1/14/2025 23:07        ModernCRM   
2       1/17/2025 1:29  AnalyticsEngine   
3  2025-07-12 00:24:16         ModernHR   
4  2025-06-02 18:25:23    BillingSystem   

                                         log_message    target_label  \
0  nova.osapi_compute.wsgi.server [req-b9718cd8-f...     HTTP Status   
1     Email service experiencing issues with sending  Critical Error   
2          Unauthorized access to data was attempted  Security Alert   
3  nova.osapi_compute.wsgi.server [req-4895c258-b...     HTTP Status   
4  nova.osapi_compute.wsgi.server [req-ee8bc8ba-9...     HTTP Status   

  complexity  cluster  
0       bert        0  
1       bert        1  
2       bert        2  
3       bert        0  
4       bert        0  


In [10]:
print(df.cluster.unique())

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135]


In [11]:
#size of each cluster
print(df['cluster'].value_counts())

cluster
0      1017
5       147
11      100
13       86
7        60
       ... 
102       1
103       1
105       1
106       1
135       1
Name: count, Length: 136, dtype: int64


In [12]:
df[df.cluster == 1].head()

,timestamp,source,log_message,target_label,complexity,cluster
1,1/14/2025 23:07,ModernCRM,Email service experiencing issues with sending,Critical Error,bert,1
10,8/9/2025 18:58,ModernCRM,Email server encountered a sending fault,Error,bert,1
217,1/22/2025 5:45,BillingSystem,Mail service encountered a delivery glitch,Error,bert,1
248,5/2/2025 23:04,ModernHR,Service disruption caused by email sending error,Critical Error,bert,1
265,3/30/2025 23:53,ModernCRM,Email system had a problem sending emails,Error,bert,1


In [13]:
import re

In [14]:
regex_patterns = {
        r"User User\d+ logged (in|out).": "User Action",
        r"Backup (started|ended) at .*": "System Notification",
        r"Backup completed successfully.": "System Notification",
        r"System updated to version .*": "System Notification",
        r"File .* uploaded successfully by user .*": "System Notification",
        r"Disk cleanup completed successfully.": "System Notification",
        r"System reboot initiated by user .*": "System Notification",
        r"Account with ID .* created by .*": "User Action"
    }

In [15]:
def classify_with_regex(log_message, regex_patterns = regex_patterns):
    for pattern, label in regex_patterns.items():
        if re.search(pattern, log_message, re.IGNORECASE):
            return label
    return "Unclassified"

In [16]:
classify_with_regex("User User123 logged OUT.")

'User Action'

In [17]:
df['regex_label'] = df['log_message'].apply(classify_with_regex)

In [18]:
#print any 5 rows which are not unclassified
print(df[df['regex_label'] != "Unclassified"].head())

          timestamp         source  \
7   10/11/2025 8:44       ModernHR   
14    1/4/2025 1:43  ThirdPartyAPI   
15    5/1/2025 9:41      ModernCRM   
18  2/22/2025 17:49      ModernCRM   
27  9/24/2025 19:57  ThirdPartyAPI   

                                          log_message         target_label  \
7   File data_6169.csv uploaded successfully by us...  System Notification   
14  File data_3847.csv uploaded successfully by us...  System Notification   
15                     Backup completed successfully.  System Notification   
18           Account with ID 5351 created by User634.          User Action   
27                           User User685 logged out.          User Action   

   complexity  cluster          regex_label  
7       regex        4  System Notification  
14      regex        4  System Notification  
15      regex        8  System Notification  
18      regex        9          User Action  
27      regex       11          User Action  


In [19]:
#print the distribution of regex labels
print(df['regex_label'].value_counts())

regex_label
Unclassified           1910
System Notification     356
User Action             144
Name: count, dtype: int64


In [20]:
df_non_regex = df[df['regex_label'] == "Unclassified"].copy()
df_non_regex.shape

(1910, 7)

In [21]:
#find rare labels in target_label column
rare_labels = df_non_regex['target_label'].value_counts()[df_non_regex['target_label'].value_counts() < 10]
print(rare_labels)

target_label
Workflow Error         4
Deprecation Warning    3
Name: count, dtype: int64


In [22]:
#find those 'source' whose target_label is in rare_labels
rare_sources = df_non_regex[df_non_regex['target_label'].isin(rare_labels.index)]['source'].unique()
print(rare_sources)

['LegacyCRM']


In [26]:
df_non_rare = df_non_regex[df_non_regex.source != rare_sources[0]]
df_non_rare.source.unique()

array(['ModernCRM', 'AnalyticsEngine', 'ModernHR', 'BillingSystem',
       'ThirdPartyAPI'], dtype=object)

In [25]:
#Apply BERT for all sources except those in rare_sources

In [31]:
filtered_embeddings = model.encode(df_non_rare['log_message'].tolist())
filtered_embeddings[0]

array([-1.02939621e-01,  3.35459411e-02, -2.20260732e-02,  1.55101740e-03,
       -9.86917876e-03, -1.78956270e-01, -6.34409785e-02, -6.01761639e-02,
        2.81109158e-02,  5.99620491e-02, -1.72618348e-02,  1.43363548e-03,
       -1.49560034e-01,  3.15287686e-03, -5.66030927e-02,  2.71685235e-02,
       -1.49891041e-02, -3.54037657e-02, -3.62936445e-02, -1.45410765e-02,
       -5.61491773e-03,  8.75539035e-02,  4.55120578e-02,  2.50963885e-02,
        1.00187510e-02,  1.24267349e-02, -1.39923573e-01,  7.68696293e-02,
        3.14095505e-02, -4.15247958e-03,  4.36902344e-02,  1.71250012e-02,
       -8.00951198e-02,  5.74006326e-02,  1.89091656e-02,  8.55262503e-02,
        3.96398641e-02, -1.34371817e-01, -1.44360063e-03,  3.06704035e-03,
        1.76854044e-01,  4.44885530e-03, -1.69274509e-02,  2.24266481e-02,
       -4.35049310e-02,  6.09034160e-03, -9.98169929e-03, -6.23972900e-02,
        1.07372422e-02, -6.04895083e-03, -7.14660808e-02, -8.45799781e-03,
       -3.18019874e-02, -

In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [38]:
X_train, X_test, y_train, y_test = train_test_split(filtered_embeddings, df_non_rare['target_label'], test_size=0.3, random_state=42)

In [39]:
clf = LogisticRegression(max_iter=1000)

In [40]:
clf.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [41]:
y_pred = clf.predict(X_test)

In [42]:
report = classification_report(y_test, y_pred)

In [43]:
print(report)

                precision    recall  f1-score   support

Critical Error       0.91      1.00      0.95        48
         Error       0.98      0.89      0.93        47
   HTTP Status       1.00      1.00      1.00       304
Resource Usage       1.00      1.00      1.00        49
Security Alert       1.00      0.99      1.00       123

      accuracy                           0.99       571
     macro avg       0.98      0.98      0.98       571
  weighted avg       0.99      0.99      0.99       571



In [44]:
import joblib

In [ ]:
joblib.dump(clf, '../models/log_classifier_model.joblib')

['models/log_classifier_model.joblib']